# European benchmark: GTFS reference-day metrics
**IE Sustainability Datathon (Kearney) · Deliverable 1 · Supporting notebook**

Computes `n_bus_lines` and `daily_bus_km` for each comparison city on a disclosed reference day, filtered to the brief's mandatory network perimeter (section 4.4). Helsinki is the fully reproducible peer. Output: `data_derived/benchmark_gtfs_intermediate.csv`, read by the main notebook, which writes `File_1.csv`. This notebook does not write any mandatory CSV.

In [ ]:
# Environment: runs as-is in Google Colab (Python 3.10+).
%pip install -q pandas numpy pyproj shapely folium plotly kaleido requests

import os, io, json, zipfile, datetime as dt, hashlib, textwrap
import numpy as np, pandas as pd, requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

BASE = "/content/datathon" if os.path.exists("/content") else os.path.abspath("..")
RAW, DERIVED, OUT = (os.path.join(BASE, d) for d in ("data_raw", "data_derived", "outputs"))
for d in (RAW, DERIVED, OUT):
    os.makedirs(d, exist_ok=True)

RUN_DATE = dt.date.today().isoformat()
import sys, platform
print(f"Run date {RUN_DATE} | Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
print("Folders:", RAW, DERIVED, OUT, sep="\n  ")

In [ ]:
# ---- Data register: every downloaded or manually obtained source is logged here.
DATA_REGISTER = []

def fetch(url, filename, note="", manual=False):
    """Download a public file into data_raw/ and log source, date, size and checksum.
    manual=True: the file was obtained by hand (registration/API); it must already be in data_raw/."""
    path = os.path.join(RAW, filename)
    if not manual and not os.path.exists(path):
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    size = os.path.getsize(path) if os.path.exists(path) else None
    sha = hashlib.sha256(open(path, "rb").read()).hexdigest()[:12] if size else None
    DATA_REGISTER.append(dict(file=filename, url=url, downloaded=RUN_DATE, bytes=size,
                              sha256_12=sha, obtained="manual" if manual else "script", note=note))
    return path

def show_register():
    display(pd.DataFrame(DATA_REGISTER))

# ---- Provenance labels (brief section 4.6): V verified, D derived, P proxy, A assumption.
ASSUMPTIONS = []

def assume(variable, value, label, evidence, impact="", sensitivity=""):
    assert label in {"V", "D", "P", "A"}, "label must be V, D, P or A"
    ASSUMPTIONS.append(dict(variable=variable, value=value, label=label, evidence=evidence,
                            impact=impact, sensitivity=sensitivity))
    return value

# ---- Visible checks: every step prints CHECK ... PASS / FAIL.
CHECKS = []

def check(name, ok, detail=""):
    CHECKS.append(dict(check=name, result="PASS" if ok else "FAIL", detail=detail))
    print(f"CHECK {'PASS' if ok else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    return ok

In [ ]:
# Fixed parameters from the official brief, section 4.5. Do not hard-code these numbers anywhere else.
P = dict(
    kwh_km      = {"Standard": 1.30, "Articulated": 1.85},   # battery energy incl. HVAC
    range_km    = {"Standard": 300,  "Articulated": 240},    # effective daily range (75% of tender minimum)
    cng_kg_km   = {"Standard": 0.45, "Articulated": 0.65},
    charger_kw  = 100,
    window_h    = 7,                                          # 23:00-06:00
    charge_eff  = 0.90,
    bus_price_eur = {"Standard": 615_000, "Articulated": 800_000},  # excl. VAT
    benchmark_kwh_km = 1.30,
    service_days = 365,
    # TODO(owner: emissions) confirm value, unit and boundary against the cited page before submission
    cng_kgco2_per_kg = assume("CO2 factor CNG", 2.7175, "V", "MITECO emission factors (inventory annex VII)", "Files 2, 4"),
    grid_gco2_kwh    = assume("CO2 factor electricity Spain 2025", 108.4, "V", "Red Electrica 2025 generation emissions", "Files 1, 2, 4"),
)
display(pd.DataFrame([(k, v) for k, v in P.items()], columns=["parameter", "value"]))

In [ ]:
# ---- Generic GTFS helpers (shared logic with the benchmark notebook).
def read_gtfs(zip_path, tables=("agency","routes","trips","stop_times","calendar","calendar_dates","shapes","feed_info")):
    feed = {}
    with zipfile.ZipFile(zip_path) as z:
        names = set(z.namelist())
        for t in tables:
            if f"{t}.txt" in names:
                feed[t] = pd.read_csv(z.open(f"{t}.txt"), dtype=str, low_memory=False)
    return feed

def active_service_ids(feed, date_str):
    """Service ids running on date_str (YYYY-MM-DD), applying calendar then calendar_dates exceptions."""
    d = pd.Timestamp(date_str); ymd = d.strftime("%Y%m%d"); dow = d.day_name().lower()
    active = set()
    if "calendar" in feed:
        c = feed["calendar"]
        m = (c["start_date"] <= ymd) & (c["end_date"] >= ymd) & (c[dow] == "1")
        active |= set(c.loc[m, "service_id"])
    if "calendar_dates" in feed:
        cd = feed["calendar_dates"]; cd = cd[cd["date"] == ymd]
        active |= set(cd.loc[cd["exception_type"] == "1", "service_id"])
        active -= set(cd.loc[cd["exception_type"] == "2", "service_id"])
    return active

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

def shape_lengths_km(feed):
    s = feed["shapes"].copy()
    for c in ("shape_pt_lat","shape_pt_lon"): s[c] = s[c].astype(float)
    s["shape_pt_sequence"] = s["shape_pt_sequence"].astype(int)
    s = s.sort_values(["shape_id","shape_pt_sequence"])
    g = s.groupby("shape_id")
    s["seg"] = haversine_km(g["shape_pt_lat"].shift(), g["shape_pt_lon"].shift(), s["shape_pt_lat"], s["shape_pt_lon"])
    return s.groupby("shape_id")["seg"].sum()

def trip_km(feed, trip_ids, dist_unit_m=None):
    """Km per trip. Preferred: shape_dist_traveled (unit must be confirmed: metres or km);
    fallback: geodesic length of the trip's shape. Prints which method each trip used."""
    st = feed["stop_times"][feed["stop_times"]["trip_id"].isin(trip_ids)]
    km = pd.Series(dtype=float)
    if "shape_dist_traveled" in st and st["shape_dist_traveled"].notna().any():
        d = st.assign(x=pd.to_numeric(st["shape_dist_traveled"], errors="coerce")).groupby("trip_id")["x"].agg(lambda v: v.max() - v.min())
        assert dist_unit_m is not None, "Confirm shape_dist_traveled unit (metres or km) and pass dist_unit_m=True/False"
        km = d / 1000 if dist_unit_m else d
    missing = set(trip_ids) - set(km.dropna().index)
    if missing and "shapes" in feed:
        lens = shape_lengths_km(feed)
        t = feed["trips"].set_index("trip_id").loc[list(missing), "shape_id"]
        km = pd.concat([km.dropna(), t.map(lens)])
    print(f"trip_km: {len(km)} trips, {len(missing)} via shape geometry fallback")
    return km

## Cities, feeds and perimeters
Feeds are those linked in the brief, section 4.4. Perimeter filters must be documented per city (operator / agency, route_type 3 = bus, exclusions).

In [ ]:
CITIES = {
    # city: dict(feed landing page, operator perimeter, reference date, filter notes)
    "Helsinki": dict(landing="https://www.hsl.fi/en/hsl/open-data", operator="HSL", ref_date="YYYY-MM-DD", filter="TODO: HSL bus routes, route_type 3 (and HSL extended bus types if used)"),
    "Berlin":   dict(landing="https://unternehmen.vbb.de/en/digital-services/datasets/", operator="BVG", ref_date="YYYY-MM-DD", filter="TODO: VBB feed, agency = BVG, bus routes only"),
    "Paris":    dict(landing="https://data.iledefrance-mobilites.fr/explore/dataset/offre-horaires-tc-gtfs-idfm/", operator="RATP (IDFM network)", ref_date="YYYY-MM-DD", filter="TODO: perimeter per brief 4.4; large feed, read stop_times in chunks"),
    "Brussels": dict(landing="https://data.belgianmobility.io/en/data.html?agency=stibmivb", operator="STIB-MIVB", ref_date="YYYY-MM-DD", filter="TODO: STIB bus routes; feed is a March 2026 archive — check validity"),
}
display(pd.DataFrame(CITIES).T)

In [ ]:
def city_metrics(city, zip_path, route_filter, dist_unit_m=None):
    """route_filter(feed) -> set of route_ids in the perimeter. Returns n_bus_lines and daily_bus_km on the city's reference day."""
    feed = read_gtfs(zip_path)
    ref = CITIES[city]["ref_date"]
    services = active_service_ids(feed, ref)
    routes = route_filter(feed)
    trips = feed["trips"][feed["trips"].service_id.isin(services) & feed["trips"].route_id.isin(routes)]
    km = trip_km(feed, set(trips.trip_id), dist_unit_m)
    lines = trips.merge(feed["routes"][["route_id", "route_short_name"]], on="route_id").route_short_name.nunique()
    out = dict(city=city, reference_date=ref, n_bus_lines=int(lines), daily_bus_km=round(float(km.sum()), 1), n_trips=len(trips))
    print(out)
    return out

## Helsinki (fully reproducible peer)

In [ ]:
# TODO(owner: Helsinki)
# fetch feed, define route_filter, confirm reference day in HSL calendar, run city_metrics, sanity-check km per line


## Berlin

In [ ]:
# TODO(owner: Berlin)
# fetch VBB feed, filter BVG bus routes, run city_metrics


## Paris

In [ ]:
# TODO(owner: Paris)
# fetch IDFM feed (large), apply brief perimeter, run city_metrics or document fallback


## Brussels

In [ ]:
# TODO(owner: Brussels)
# fetch STIB feed, check validity of the March 2026 archive, run city_metrics or document fallback


## Output
One row per city; fields not computable from GTFS are left for the main notebook (fleet, targets, charging model, grid factor).

In [ ]:
# results = [helsinki, berlin, paris, brussels]
# out = pd.DataFrame(results); out.to_csv(os.path.join(DERIVED, "benchmark_gtfs_intermediate.csv"), index=False)
# display(out); show_register()